## 04 - Test the /predict endpoint

Load a real experiment from the training CSV, reshape it into the API payload
(timestamps + Z/W/X values dict), POST it to the running FastAPI server, and
compare the predicted titer to the recorded final titer.

Start the server (from the project root):

```bash
uv run -m uvicorn src.api:app --reload
```

In [1]:
import pandas as pd
import requests

In [2]:
API_URL = "http://localhost:8000"
TRAIN_DATA_PATH = "../data/datahow_interview_train_data.csv"
TRAIN_TARGETS_PATH = "../data/datahow_interview_train_targets.csv"
EXP = "Exp 1"

In [3]:
requests.get(f"{API_URL}/health").json()

{'status': 'ok'}

### Load the data and pick one experiment

In the raw CSV, `Z:` setpoints are only filled on day 0 and `NaN` everywhere
else; `W:` and `X:` columns are populated at every timestep.

In [4]:
train_df = pd.read_csv(TRAIN_DATA_PATH).drop(columns="RowID")
targets_df = pd.read_csv(TRAIN_TARGETS_PATH).drop(columns="RowID")

exp_df = train_df.query("Exp == @EXP").sort_values("Time[day]").reset_index(drop=True)
exp_df.head()

,Exp,Time[day],Z:FeedStart,Z:FeedEnd,Z:FeedRateGlc,Z:FeedRateGln,Z:phStart,Z:phEnd,Z:phShift,Z:tempStart,...,W:temp,W:pH,W:FeedGlc,W:FeedGln,X:VCD,X:Glc,X:Gln,X:Amm,X:Lac,X:Lysed
0,Exp 1,0,2.0,10.0,5.656566,6.818182,6.808081,6.479798,9.0,37.282828,...,37.282828,6.808081,0.000000,0.000000,1.682644,2.853045,6.005418,0.100000,0.100000,0.000000
1,Exp 1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.282828,6.808081,0.000000,0.000000,3.617721,2.290199,3.598475,0.324724,1.348831,0.002831
2,Exp 1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.282828,6.808081,5.656566,6.818182,6.431879,0.752652,0.481756,0.687781,3.065268,0.000884
3,Exp 1,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.282828,6.808081,5.656566,6.818182,10.279057,4.276583,2.516062,0.972871,4.196027,0.000000
4,Exp 1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.282828,6.808081,5.656566,6.818182,14.888965,7.350100,3.471558,1.322927,4.845310,0.000218


### Build the request payload

Per `inference_server_spec.yml`:
- `Z:` keys must be single-element arrays (we take the day-0 value)
- `W:` and `X:` keys must be arrays whose length matches `timestamps`

In [5]:
z_cols = [c for c in exp_df.columns if c.startswith("Z:")]
w_cols = [c for c in exp_df.columns if c.startswith("W:")]
x_cols = [c for c in exp_df.columns if c.startswith("X:")]

day0 = exp_df.loc[exp_df["Time[day]"] == 0].iloc[0]

values = {col: [float(day0[col])] for col in z_cols}
for col in w_cols + x_cols:
    values[col] = exp_df[col].astype(float).tolist()

payload = {
    "timestamps": exp_df["Time[day]"].astype(float).tolist(),
    "values": values,
}

print(f"timestamps: {payload['timestamps']}")
print(f"# Z keys: {len(z_cols)}, # W keys: {len(w_cols)}, # X keys: {len(x_cols)}")

timestamps: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
# Z keys: 13, # W keys: 4, # X keys: 6


### Call the endpoint and compare to the recorded titer

In [6]:
response = requests.post(f"{API_URL}/predict", json=payload)
response.raise_for_status()
result = response.json()
result

{'titer': 2438.5352989273506}

In [7]:
actual = float(targets_df.loc[targets_df["Exp"] == EXP, "Y:Titer"].iloc[0])
predicted = float(result["titer"])
print(f"Experiment: {EXP}")
print(f"  predicted titer: {predicted:.2f}")
print(f"  actual titer:    {actual:.2f}")
print(f"  abs error:       {abs(predicted - actual):.2f}")
print(f"  rel error:       {abs(predicted - actual) / actual:.2%}")

Experiment: Exp 1
  predicted titer: 2438.54
  actual titer:    2550.10
  abs error:       111.56
  rel error:       4.37%
